# Chemistry PDF extractor (Kaggle)

**Code**: public GitHub (cloned below). **PDFs**: Kaggle dataset only.

Notebook settings: **GPU** + **Internet**.

PDFs may live in subfolders inside the dataset zip — the notebook searches recursively.

In [ ]:
!apt-get update -qq && apt-get install -y -qq poppler-utils git

In [ ]:
import os
import sys
from pathlib import Path

GITHUB_REPO = "https://github.com/YOUR_USERNAME/test-paper-extractor.git"
GITHUB_BRANCH = "main"
PDF_INPUT_DIR = Path("/kaggle/input/cbse_12_checm_samples")

PROJECT_ROOT = Path("/kaggle/working/test-paper-extractor")
OUT_ROOT = Path("/kaggle/working/out")
DPI = 300
QWEN_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"

print("Datasets mounted:", [p.name for p in Path("/kaggle/input").iterdir()])

In [ ]:
import shutil
import subprocess

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

subprocess.check_call([
    "git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
    GITHUB_REPO, str(PROJECT_ROOT),
])

if not (PROJECT_ROOT / "extract_paper.py").is_file():
    raise FileNotFoundError(f"Repo cloned but extract_paper.py missing under {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from pdf_discovery import discover_pdfs, format_tree_hint

pdfs = discover_pdfs(PDF_INPUT_DIR, recursive=True)
if not pdfs:
    raise FileNotFoundError(
        f"No PDFs under {PDF_INPUT_DIR}.\n"
        f"Check PDF_INPUT_DIR matches Add Data slug. Files seen:\n  {format_tree_hint(PDF_INPUT_DIR)}"
    )

print(f"Cloned {GITHUB_REPO}")
print(f"Found {len(pdfs)} PDF(s):")
for p in pdfs:
    print(" ", p.relative_to(PDF_INPUT_DIR))

!pip install -q -r requirements.txt

In [ ]:
from batch_extract import main as batch_main

OUT_ROOT.mkdir(parents=True, exist_ok=True)
argv = [
    str(PDF_INPUT_DIR),
    "--out-root", str(OUT_ROOT),
    "--recursive",
    "--",
    "--backend", "qwen_hf",
    "--dpi", str(DPI),
    "--qwen-model", QWEN_MODEL,
    "--save-page-images",
]
batch_main(argv)

In [ ]:
from preview import build_preview_html

for questions_path in OUT_ROOT.glob("*/questions.json"):
    print("Preview:", build_preview_html(questions_path))

In [ ]:
import shutil

zip_path = Path("/kaggle/working/out.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", OUT_ROOT)
print("Download from Output:", zip_path)